# Datavidia 11.0 shipment classification
Use **Runtime → Change runtime type → GPU**. Run the setup cells in order. Keep the competition files in your approved data storage. This notebook uses a private Google Drive folder for data and run artifacts.

In [ ]:
from pathlib import Path
import base64
import os
import re
import subprocess
import sys

REPO = "YOUR_GITHUB_USERNAME/arkavidia"
BRANCH = "main"
PRIVATE_REPO = True

if REPO.startswith("YOUR_") or not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPO):
    raise ValueError("Set REPO to your GitHub owner and repository name.")

env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
if PRIVATE_REPO:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    encoded = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
               GIT_CONFIG_VALUE_0=f"Authorization: Basic {encoded}")
    del token, encoded

repo_url = f"https://github.com/{REPO}.git"
project = Path("/content/arkavidia")
if project.exists():
    remote = subprocess.check_output(
        ["git", "-C", str(project), "remote", "get-url", "origin"], text=True
    ).strip()
    if remote != repo_url:
        raise ValueError("Existing runtime checkout points to a different repository. Start a fresh runtime.")
    subprocess.run(["git", "-C", str(project), "pull", "--ff-only", "origin", BRANCH], env=env, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, repo_url, str(project)],
                   env=env, check=True)
del env
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=project, check=True)

For a private repository, create a GitHub fine-grained token scoped to that repository with **Contents: Read-only**. Add it under **Secrets** in Colab as `GITHUB_TOKEN` and turn on notebook access. GitHub's notebook picker opens the notebook. The token separately lets the runtime clone the project code.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
storage = Path("/content/drive/MyDrive/arkavidia")
data = storage / "data"
data.mkdir(parents=True, exist_ok=True)
config = storage / "config" / "colab.toml"
config.parent.mkdir(parents=True, exist_ok=True)
if not config.exists():
    shutil.copyfile(project / "config" / "default.toml", config)
print("Dataset directory:", data)
print("Edit this TOML after inspecting the files:", config)

def pipeline(*args):
    subprocess.run([sys.executable, "run.py", "--config", str(config), *args], cwd=project, check=True)

pipeline("inspect")

Put the extracted competition files in `MyDrive/arkavidia/data/` if permitted by the competition rules, then rerun the inspect cell. Edit `MyDrive/arkavidia/config/colab.toml` to match the filenames, ID, target, prediction-time features, and any auxiliary tables. The default `root = "../data"` already points to the data folder. See the project README for relation and date examples.

In [ ]:
from datetime import datetime, timezone

subprocess.run(["nvidia-smi", "-L"], check=True)
run_root = storage / "outputs"
run_root.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run = run_root / run_id
pipeline("train", "--profile", "baseline", "--device", "gpu", "--out", str(run))
print("Submission:", run / "submission.csv")
print("Metrics:", run / "metrics.json")

In [ ]:
from google.colab import files
files.download(str(run / "submission.csv"))